# FlashAttention: IO-Aware Attention - 실습 코드 1: FlashAttention 사용법 (PyTorch)

- Tutorial ID: `expand-flash-attention`
- Tutorial: FlashAttention: IO-Aware Attention
- Section ID: `expand-flash-attention-code-1`
- Section: 실습 코드 1: FlashAttention 사용법 (PyTorch)


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: FlashAttention 사용법 (PyTorch)
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import torch
from torch.nn.functional import scaled_dot_product_attention

# PyTorch 2.0+ 에서 FlashAttention 자동 사용
q = torch.randn(2, 8, 1024, 64, device='cuda', dtype=torch.float16)
k = torch.randn(2, 8, 1024, 64, device='cuda', dtype=torch.float16)
v = torch.randn(2, 8, 1024, 64, device='cuda', dtype=torch.float16)

# FlashAttention 자동 선택 (seq_len > 256일 때)
output = scaled_dot_product_attention(q, k, v, is_causal=True)

print(f"Output shape: {output.shape}")
# 메모리 비교
import torch.cuda as cuda
cuda.reset_peak_memory_stats()
_ = scaled_dot_product_attention(q, k, v, is_causal=True)
print(f"Peak memory: {cuda.max_memory_allocated()/1e6:.1f} MB")